# SEN12MS-CR 학습 노트북

이 노트북은 `main.py`와 같은 학습 흐름을 그대로 쓰되, 배치 확인과 결과 시각화만 노트북에 맞게 덧붙인 버전이다.

Colab에서는 저장소가 없으면 레포를 먼저 clone하고, 의존성 설치는 `uv`로 맞춘 뒤 같은 `main.py` helper를 재사용한다.

- 실행 환경 준비
- Colab 저장소 준비 및 `uv` 설치
- 학습 설정값 지정
- `main.py` helper 재사용
- 배치 shape 확인
- 학습 로그 표/그래프 확인
- 복원 결과 예시 저장 및 시각화


In [ ]:
import os
import subprocess
import sys
from pathlib import Path

# Colab에서는 노트북만 단독으로 열릴 수 있어서,
# 저장소가 없으면 먼저 레포를 clone한 뒤 같은 러너 코드를 재사용한다.
IN_COLAB = "google.colab" in sys.modules
HF_TOKEN_INPUT = ""
REPO_URL = "https://github.com/smturtle2/cr-project.git"
REPO_DIR = Path("/content/cr-project")

# 노트북 환경에는 uv가 없는 경우가 많으므로 가장 먼저 설치한다.
# bootstrap만 pip로 하고, 실제 의존성 설치는 아래에서 uv로 처리한다.
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)

# Colab에서 현재 작업 디렉터리에 저장소가 없으면 레포를 clone해서 사용한다.
# 이렇게 해야 아래 셀에서 `import main as train_app`가 항상 같은 방식으로 동작한다.
if IN_COLAB and not (Path.cwd() / "pyproject.toml").exists():
    if not REPO_DIR.exists():
        subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
    os.chdir(REPO_DIR)

WORKDIR = Path.cwd()
if not (WORKDIR / "pyproject.toml").exists():
    raise FileNotFoundError("main.ipynb는 저장소 루트에서 실행해야 합니다.")

# 설치는 항상 uv를 통해 현재 레포 기준 editable install로 맞춘다.
# Colab에서도 이 경로를 쓰면 uv.lock과 pyproject 기준이 그대로 유지된다.
subprocess.run(
    [
        "uv",
        "pip",
        "install",
        "--system",
        "--no-cache",
        "-e",
        ".",
    ],
    check=True,
)

# 결과물은 로컬에서는 현재 작업 디렉터리 아래에,
# Colab에서는 Drive가 있으면 Drive 아래에 저장해서 세션이 끊겨도 남도록 한다.
if IN_COLAB and Path("/content/drive/MyDrive").exists():
    OUTPUT_DIR = Path("/content/drive/MyDrive/cr-project/artifacts")
else:
    OUTPUT_DIR = Path("/content/artifacts") if IN_COLAB else WORKDIR / "artifacts"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# 토큰은 환경 변수 우선, 없으면 셀 상단 문자열 입력값을 사용한다.
token = os.environ.get("HF_TOKEN", "").strip() or HF_TOKEN_INPUT.strip()
if token:
    os.environ["HF_TOKEN"] = token

print(f"IN_COLAB={IN_COLAB}")
print(f"WORKDIR={WORKDIR}")
print(f"OUTPUT_DIR={OUTPUT_DIR}")
print(f"HF_TOKEN={'configured' if token else 'not found'}")


In [ ]:
import importlib
from pathlib import Path

import pandas as pd
import torch
from IPython.display import display
from torch import nn
from cr_train import Trainer

# 긴 보조 함수는 notebook에 다시 복붙하지 않고 main.py에서 그대로 가져온다.
# 이렇게 두면 스크립트와 노트북이 같은 로직을 공유하므로 동작 차이가 줄어든다.
import main as train_app
train_app = importlib.reload(train_app)

# 아래 값들만 바꾸면 같은 helper를 써서 다양한 실험을 반복할 수 있다.
batch_size = 8
seed = 42
max_epochs = 10
train_max_samples = 2048
val_max_samples = 512
test_max_samples = 512
resume = None
run_test = False
num_examples = 4
example_stage = "val"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device={device}")
print(f"output_dir={OUTPUT_DIR}")


In [ ]:
# main.py는 학습 실행 helper를 제공하고,
# 노트북은 표와 그래프처럼 화면에 바로 보여 줄 부분만 따로 둔다.


def build_model() -> nn.Module:
    # 실제 실험에 사용할 모델을 여기서 반환하면 된다.
    # cr-train 규약상 `forward(sar, cloudy)` 시그니처만 맞추면 된다.
    raise NotImplementedError("build_model()을 구현하세요.")



def build_optimizer(model: nn.Module) -> torch.optim.Optimizer:
    # 모델 파라미터를 받아 optimizer를 구성한다.
    raise NotImplementedError("build_optimizer()를 구현하세요.")



def build_loss():
    # loss도 notebook에서 교체할 수 있는 확장 포인트로 따로 둔다.
    # Trainer 계약에 맞춰 `(prediction, batch)` 형태의 함수를 반환한다.
    l1_loss = nn.L1Loss()

    def loss_fn(prediction: torch.Tensor, batch: dict[str, object]) -> torch.Tensor:
        return l1_loss(prediction, batch["target"])

    return loss_fn



def build_metrics() -> dict[str, object]:
    # metric도 모델/optimizer와 같은 수준의 교체 지점으로 유지한다.
    def mae(prediction: torch.Tensor, batch: dict[str, object]) -> torch.Tensor:
        return torch.mean(torch.abs(prediction - batch["target"]))

    return {"mae": mae}



def style_frame(frame: pd.DataFrame, caption: str):
    # epoch 번호와 step은 정수로 두고,
    # 나머지는 성격에 맞는 포맷을 적용해서 표를 읽기 쉽게 만든다.
    formats = {}
    for col in frame.columns:
        if col in {"epoch", "global_step"}:
            continue
        if col == "elapsed_sec":
            formats[col] = "{:.2f}s"
        else:
            formats[col] = "{:.4f}"

    style = frame.style.hide(axis="index").set_caption(caption)
    return style.format(formats)



def plot_history_df(history_df: pd.DataFrame) -> None:
    # loss, 일반 metric, epoch 시간대를 분리해서 그리면
    # 값 스케일이 섞이지 않아 추세를 읽기 쉽다.
    if history_df.empty:
        return

    import matplotlib.pyplot as plt

    metric_cols = [col for col in history_df.columns if col not in {"epoch", "global_step"}]
    loss_cols = [col for col in metric_cols if col.endswith("_loss")]
    time_cols = [col for col in metric_cols if col == "elapsed_sec"]
    other_cols = [col for col in metric_cols if col not in loss_cols and col not in time_cols]
    groups = [keys for keys in (loss_cols, other_cols, time_cols) if keys]
    if not groups:
        return

    fig, axes = plt.subplots(len(groups), 1, figsize=(10, 4 * len(groups)), sharex=True)
    if len(groups) == 1:
        axes = [axes]

    epochs = history_df["epoch"]
    colors = plt.get_cmap("tab10")

    for ax, keys in zip(axes, groups):
        for index, key in enumerate(keys):
            ax.plot(
                epochs,
                history_df[key],
                label=key,
                color=colors(index % 10),
                linewidth=2.2,
                marker="o",
                markersize=5,
            )
        ax.grid(True, alpha=0.25)
        ax.legend(frameon=False)
        ax.spines["top"].set_visible(False)
        ax.spines["right"].set_visible(False)
        ax.set_ylabel("seconds" if keys == time_cols else "value")

    axes[0].set_title("training history")
    axes[-1].set_xlabel("epoch")
    plt.show()



def show_history(history: list[dict[str, int | float]]) -> pd.DataFrame:
    # Trainer summary를 표와 그래프로 한 번에 확인하기 위한 notebook 전용 helper다.
    # 최신 cr-train이 주는 elapsed_sec도 함께 보여 줘서 epoch 시간 변화를 바로 볼 수 있게 한다.
    history_df = pd.DataFrame(history)
    display(style_frame(history_df, "history"))
    plot_history_df(history_df)

    best_key = "val_loss" if "val_loss" in history_df.columns else "train_loss"
    best_row = history_df.loc[[history_df[best_key].idxmin()]]
    display(style_frame(best_row, f"best row by {best_key}"))
    return history_df



def show_test_metrics(summary: dict[str, object]) -> pd.DataFrame:
    # test 결과는 막대그래프로도 보여 줘서 metric 상대 크기를 바로 볼 수 있게 한다.
    import matplotlib.pyplot as plt

    row = {}
    if "loss" in summary:
        row["test_loss"] = float(summary["loss"])
    for name, value in summary.get("metrics", {}).items():
        row[f"test_{name}"] = float(value)

    test_df = pd.DataFrame([row])
    display(style_frame(test_df, "test metrics"))

    series = test_df.iloc[0].sort_values()
    fig, ax = plt.subplots(figsize=(8, max(2.5, 0.7 * len(series))))
    ax.barh(series.index, series.values, color="#4C78A8")
    ax.set_title("test metrics")
    ax.set_xlabel("value")
    ax.grid(True, axis="x", alpha=0.25)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)
    plt.show()
    return test_df


In [ ]:
# 모델 초기화 재현성을 위해 Trainer 생성 전에 시드를 먼저 고정한다.
train_app.seed_everything(seed)
train_app.print_hf_auth_status()

# 모델과 optimizer는 프로젝트 쪽 구현만 연결하면 된다.
model = build_model().to(device)
optimizer = build_optimizer(model)

# Trainer 생성은 직접 두되, loss와 metric은 build 함수로 분리해서
# 프로젝트별 교체 지점이 어디인지 노트북에서도 똑같이 보이게 한다.
# 그래서 thin wrapper는 없애고 build 쪽만 남긴다.
# Trainer가 어떤 설정으로 생성되는지 셀에서 바로 보이도록 여기서 직접 만든다.
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    loss=build_loss(),
    metrics=build_metrics(),
    max_train_samples=train_max_samples,
    max_val_samples=val_max_samples,
    max_test_samples=test_max_samples,
    batch_size=batch_size,
    epochs=max_epochs,
    seed=seed,
    output_dir=OUTPUT_DIR,
)

# checkpoint에서 이어서 시작할 때도 main.py helper를 그대로 쓴다.
if resume is not None:
    train_app.load_checkpoint(trainer, Path(resume), device=device)

# 첫 train batch를 한 번 확인해 두면 모델 입력 shape를 바로 검증할 수 있다.
train_loader = train_app.build_loader(
    trainer,
    split="train",
    max_samples=train_max_samples,
    training=True,
    epoch_index=trainer.current_epoch,
)
sample_batch = next(iter(train_loader))

sar = sample_batch["sar"]
cloudy = sample_batch["cloudy"]
target = sample_batch["target"]
metadata = sample_batch.get("meta", {})

print(f"sar     shape={tuple(sar.shape)}  dtype={sar.dtype}  range=[{sar.min():.3f}, {sar.max():.3f}]")
print(f"cloudy  shape={tuple(cloudy.shape)}  dtype={cloudy.dtype}  range=[{cloudy.min():.3f}, {cloudy.max():.3f}]")
print(f"target  shape={tuple(target.shape)}  dtype={target.dtype}  range=[{target.min():.3f}, {target.max():.3f}]")
print(f"metadata keys: {list(metadata.keys())}")


In [ ]:
# Trainer 객체 상태를 셀에서 바로 확인할 수 있게 따로 남겨 둔다.
trainer


In [ ]:
# Trainer.step()은 한 번 호출할 때 한 epoch만 처리하므로 현재 epoch가 끝날 때까지 반복한다.
# 이때 main.py helper가 elapsed_sec도 함께 flatten해서 표와 그래프에 포함한다.
history = []
while trainer.current_epoch < trainer.epochs:
    record = trainer.step()
    history.append(train_app.flatten_record(record, global_step=trainer.global_step))

# CLI와 같은 파일 결과물도 남기고, notebook에서는 표와 그래프로 바로 확인한다.
train_app.save_history_plot(history, OUTPUT_DIR / "history.png")
history_df = show_history(history) if history else pd.DataFrame()

# 필요할 때만 test split 평가를 추가로 실행한다.
if run_test:
    test_summary = trainer.test()
    test_df = show_test_metrics(test_summary)
else:
    test_df = None

# 복원 예시는 validation 또는 test split에서 다시 loader를 만들어 저장한다.
preview_loader = train_app.build_loader(
    trainer,
    split="validation" if example_stage == "val" else "test",
    max_samples=val_max_samples if example_stage == "val" else test_max_samples,
    training=False,
    epoch_index=max(trainer.current_epoch - 1, 0),
)
train_app.save_restoration_examples(
    model=trainer.model,
    dataloader=preview_loader,
    device=device,
    output_dir=OUTPUT_DIR / "examples",
    num_examples=num_examples,
    stage=example_stage,
)
